In [ ]:
import boto3
import os
import zipfile
from datetime import datetime
from botocore.exceptions import ClientError

# --- CONFIGURACIÓN ---
BUCKET_NAME = 'bucket_name' # Reemplaza con tu bucket real
SOURCE_FOLDER = './mis_archivos/'  # Carpeta a respaldar

# Inicializar el cliente de S3
s3_client = boto3.client('s3')

In [8]:
def compress_folder(folder_path, output_zip_name):
    """
    Comprime todo el contenido de una carpeta en un archivo .zip
    """
    try:
        with zipfile.ZipFile(output_zip_name, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, dirs, files in os.walk(folder_path):
                for file in files:
                    # Crear la ruta completa y la ruta relativa dentro del zip
                    file_path = os.path.join(root, file)
                    arcname = os.path.relpath(file_path, folder_path)
                    zipf.write(file_path, arcname)
        print(f"Carpeta comprimida exitosamente: {output_zip_name}")
        return True
    except Exception as e:
        print(f"Error al comprimir: {e}")
        return False

In [9]:
def run_zipped_backup(source_dir, bucket):
    """
    Comprime una carpeta local y la sube a S3 con una estructura de fecha.
    """
    # 1. Generar nombres basados en tiempo
    timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M")
    zip_filename = f"backup_{timestamp}.zip"
    s3_key = f"backups/archives/{timestamp}/{zip_filename}"
    
    if not os.path.exists(source_dir):
        print(f"La carpeta '{source_dir}' no existe.")
        return

    # 2. Comprimir
    if compress_folder(source_dir, zip_filename):
        try:
            print(f"Subiendo backup comprimido a S3...")
            s3_client.upload_file(zip_filename, bucket, s3_key)
            print(f"Backup completado: s3://{bucket}/{s3_key}")
            
            # 3. Limpieza: Eliminar el zip local después de subirlo
            os.remove(zip_filename)
            print(f"Archivo temporal local eliminado.")
            
        except ClientError as e:
            print(f"Error de AWS: {e}")
        except Exception as e:
            print(f"Error inesperado: {e}")
    else:
        print("El proceso se detuvo porque falló la compresión.")

# --- EJECUCIÓN ---
run_zipped_backup(SOURCE_FOLDER, BUCKET_NAME)

Carpeta comprimida exitosamente: backup_2026-04-13_21-34.zip
Subiendo backup comprimido a S3...
Backup completado: s3://s3-sync-project-vv/backups/archives/2026-04-13_21-34/backup_2026-04-13_21-34.zip
Archivo temporal local eliminado.
